In [81]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Übungseinheit 1 – Datenverständnis und Vorbereitung

In [82]:
df = pd.read_csv('data/gebaeude_energie_ml.csv', delimiter=',')
df.describe()

,gebaeude_id,wohnflaeche_m2,baujahr,renovierungsjahr,anzahl_personen,isolationsstandard,avg_innentemp_c,heizgradtage,warmwasseranteil,lueftungsanlage,eta_heizung,pv_kwp,smart_meter,jahresenergieverbrauch_kwh,verbrauch_kwh_pro_m2
count,2500.00000,2500.000000,2500.00000,1637.000000,2500.000000,2500.000000,2450.000000,2500.000000,2459.000000,2500.000000,2453.000000,2436.000000,2500.000000,2500.000000,2500.000000
mean,1250.50000,187.909320,1986.78040,2003.100183,3.184800,1.948400,20.470531,3112.494400,0.180216,0.274400,1.480803,2.109741,0.459600,16735.745600,88.121920
std,721.83216,84.172807,21.63333,20.348906,1.720184,0.796236,1.323725,379.864435,0.048939,0.446301,1.029495,3.500460,0.498465,9238.177952,23.189174
min,1.00000,40.400000,1950.00000,1952.000000,1.000000,1.000000,15.400000,1811.000000,0.050000,0.000000,0.670000,0.000000,0.000000,1979.000000,25.440000
25%,625.75000,126.600000,1968.00000,1987.000000,2.000000,1.000000,19.600000,2846.000000,0.147000,0.000000,0.870000,0.000000,0.000000,10004.500000,71.887500
50%,1250.50000,175.000000,1987.00000,2008.000000,3.000000,2.000000,20.500000,3121.000000,0.180000,0.000000,0.910000,0.000000,0.000000,14930.000000,86.960000
75%,1875.25000,231.000000,2005.00000,2025.000000,4.000000,2.000000,21.400000,3360.250000,0.214000,1.000000,2.420000,4.260000,1.000000,21253.250000,103.642500
max,2500.00000,627.700000,2024.00000,2025.000000,8.000000,4.000000,24.900000,4200.000000,0.350000,1.000000,4.500000,16.000000,1.000000,75734.000000,185.410000


## Fehlende Werte finden

In [83]:
fehlende_Werte = pd.DataFrame({
    'fehlend': df.isna().sum(),
    'prozent': (df.isna().mean() * 100).round(2)
}).query('fehlend > 0').sort_values('fehlend', ascending=False)

print(fehlende_Werte)

                  fehlend  prozent
renovierungsjahr      863    34.52
pv_kwp                 64     2.56
avg_innentemp_c        50     2.00
eta_heizung            47     1.88
warmwasseranteil       41     1.64


# Daten für maschinelles Lernen vorbereiten

In [84]:
# Features vorbereiten (ohne Zielspalten und abgeleitete Spalten)
X = df.drop(columns=['jahresenergieverbrauch_kwh', 'effizienzklasse', 'verbrauch_kwh_pro_m2'], errors='ignore')

# Zielvariable für Regression
y_regression = df['jahresenergieverbrauch_kwh']

# Zielvariable für Klassifikation
y_klassifikation = df['effizienzklasse']

# Train/Test-Split für Regression (80/20)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_regression, test_size=0.2, random_state=42
)

# Train/Test-Split für Klassifikation (80/20)
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(
    X, y_klassifikation, test_size=0.2, random_state=42
)

print(f"Regression - Training: {X_train_reg.shape}, Test: {X_test_reg.shape}")
print(f"Klassifikation - Training: {X_train_class.shape}, Test: {X_test_class.shape}")

Regression - Training: (2000, 17), Test: (500, 17)
Klassifikation - Training: (2000, 17), Test: (500, 17)


In [85]:
# Kopie der Daten erstellen
X_train_reg_prep = X_train_reg.copy()
X_test_reg_prep = X_test_reg.copy()
X_train_class_prep = X_train_class.copy()
X_test_class_prep = X_test_class.copy()

# Kategoriale und numerische Spalten identifizieren
categorical_cols = X_train_reg_prep.select_dtypes(include=['object']).columns.tolist()
numerical_cols = X_train_reg_prep.select_dtypes(include=['int64', 'float64']).columns.tolist()

# 1. Fehlende Werte behandeln
# Numerische Spalten: Mit Median auffüllen
imputer_num = SimpleImputer(strategy='median')
X_train_reg_prep[numerical_cols] = imputer_num.fit_transform(X_train_reg_prep[numerical_cols])
X_test_reg_prep[numerical_cols] = imputer_num.transform(X_test_reg_prep[numerical_cols])
X_train_class_prep[numerical_cols] = imputer_num.transform(X_train_class_prep[numerical_cols])
X_test_class_prep[numerical_cols] = imputer_num.transform(X_test_class_prep[numerical_cols])

# Kategoriale Spalten: Mit häufigstem Wert auffüllen
imputer_cat = SimpleImputer(strategy='most_frequent')
X_train_reg_prep[categorical_cols] = imputer_cat.fit_transform(X_train_reg_prep[categorical_cols])
X_test_reg_prep[categorical_cols] = imputer_cat.transform(X_test_reg_prep[categorical_cols])
X_train_class_prep[categorical_cols] = imputer_cat.transform(X_train_class_prep[categorical_cols])
X_test_class_prep[categorical_cols] = imputer_cat.transform(X_test_class_prep[categorical_cols])

# 2. Kategoriale Variablen encodieren
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X_train_reg_prep[col] = le.fit_transform(X_train_reg_prep[col])
    X_test_reg_prep[col] = le.transform(X_test_reg_prep[col])
    X_train_class_prep[col] = le.transform(X_train_class_prep[col])
    X_test_class_prep[col] = le.transform(X_test_class_prep[col])
    label_encoders[col] = le

# 3. Numerische Features skalieren
scaler = StandardScaler()
X_train_reg_scaled = scaler.fit_transform(X_train_reg_prep)
X_test_reg_scaled = scaler.transform(X_test_reg_prep)
X_train_class_scaled = scaler.transform(X_train_class_prep)
X_test_class_scaled = scaler.transform(X_test_class_prep)

# Als DataFrames zurück konvertieren
X_train_reg_scaled = pd.DataFrame(X_train_reg_scaled, columns=X_train_reg_prep.columns, index=X_train_reg_prep.index)
X_test_reg_scaled = pd.DataFrame(X_test_reg_scaled, columns=X_test_reg_prep.columns, index=X_test_reg_prep.index)
X_train_class_scaled = pd.DataFrame(X_train_class_scaled, columns=X_train_class_prep.columns, index=X_train_class_prep.index)
X_test_class_scaled = pd.DataFrame(X_test_class_scaled, columns=X_test_class_prep.columns, index=X_test_class_prep.index)

print(f"Kategoriale Spalten encodiert: {categorical_cols}")
print(f"Numerische Spalten skaliert: {numerical_cols}")
print(f"\nVerbleibende fehlende Werte: {X_train_reg_scaled.isna().sum().sum()}")

Kategoriale Spalten encodiert: ['region', 'gebaeudetyp', 'fenster_typ', 'heizungsart']
Numerische Spalten skaliert: ['gebaeude_id', 'wohnflaeche_m2', 'baujahr', 'renovierungsjahr', 'anzahl_personen', 'isolationsstandard', 'avg_innentemp_c', 'heizgradtage', 'warmwasseranteil', 'lueftungsanlage', 'eta_heizung', 'pv_kwp', 'smart_meter']

Verbleibende fehlende Werte: 0


In [86]:
X_train_reg_scaled

,gebaeude_id,region,gebaeudetyp,wohnflaeche_m2,baujahr,renovierungsjahr,anzahl_personen,isolationsstandard,fenster_typ,heizungsart,avg_innentemp_c,heizgradtage,warmwasseranteil,lueftungsanlage,eta_heizung,pv_kwp,smart_meter
2055,1.118371,0.376000,-0.555044,0.401837,0.794042,0.186854,1.052147,0.055777,0.906569,-0.582366,-0.440753,0.504654,0.047280,-0.617426,-0.511490,-0.597676,1.083473
1961,0.988608,1.256564,1.947033,0.527445,1.072353,1.221383,0.472452,1.309189,-0.430554,-0.582366,-1.044990,0.491444,0.273058,1.619626,-0.511490,1.455758,1.083473
1864,0.854704,0.376000,-0.555044,1.501496,-1.154129,0.186854,2.211538,-1.197636,0.906569,-1.253682,0.692191,-0.742375,0.026755,1.619626,-0.579669,-0.597676,1.083473
2326,1.492474,-0.504563,-0.555044,-0.697821,-0.087273,0.186854,0.472452,0.055777,-1.767677,-0.582366,0.767721,-1.246999,1.401947,-0.617426,-0.599148,-0.597676,-0.922958
461,-1.082072,1.256564,1.947033,1.465947,0.979582,1.221383,-0.107244,1.309189,-0.430554,-1.253682,0.994309,-0.029032,0.745139,-0.617426,-0.560189,3.195273,-0.922958
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1638,0.542722,-0.504563,-1.389070,-0.537849,-0.643894,0.004290,-1.266635,-1.197636,0.906569,0.088949,-0.591812,-0.309085,0.868290,-0.617426,-0.599148,-0.597676,1.083473
1095,-0.206864,1.256564,-1.389070,-0.381432,0.330192,0.186854,-0.107244,-1.197636,-0.430554,-1.253682,-1.649227,-0.190194,1.176169,-0.617426,-0.492010,-0.597676,1.083473
1130,-0.158549,-1.385126,-0.555044,-1.401697,1.628973,1.221383,-1.266635,0.055777,-0.430554,1.431581,0.994309,0.200823,0.088330,-0.617426,1.115070,1.389518,1.083473
1294,0.067846,-1.385126,-1.389070,0.585509,-0.736664,-1.395366,-0.686939,-1.197636,0.906569,1.431581,-1.120519,1.331604,0.437260,-0.617426,2.731889,1.297358,-0.922958


# Übungseinheit 2 – Regressionsaufgabe

In [87]:
# 1. Lineares Modell: Linear Regression
linear_model = LinearRegression()
linear_model.fit(X_train_reg_scaled, y_train_reg)

# Vorhersagen
y_pred_linear_train = linear_model.predict(X_train_reg_scaled)
y_pred_linear_test = linear_model.predict(X_test_reg_scaled)

# Metriken für Linear Regression
linear_metrics = {
    'MAE_train': mean_absolute_error(y_train_reg, y_pred_linear_train),
    'MAE_test': mean_absolute_error(y_test_reg, y_pred_linear_test),
    'RMSE_train': np.sqrt(mean_squared_error(y_train_reg, y_pred_linear_train)),
    'RMSE_test': np.sqrt(mean_squared_error(y_test_reg, y_pred_linear_test)),
    'R2_train': r2_score(y_train_reg, y_pred_linear_train),
    'R2_test': r2_score(y_test_reg, y_pred_linear_test)
}

print("Linear Regression Ergebnisse:")
print(linear_metrics)

# 2. Nicht-lineares Modell: Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
rf_model.fit(X_train_reg_scaled, y_train_reg)

# Vorhersagen
y_pred_rf_train = rf_model.predict(X_train_reg_scaled)
y_pred_rf_test = rf_model.predict(X_test_reg_scaled)

# Metriken für Random Forest
rf_metrics = {
    'MAE_train': mean_absolute_error(y_train_reg, y_pred_rf_train),
    'MAE_test': mean_absolute_error(y_test_reg, y_pred_rf_test),
    'RMSE_train': np.sqrt(mean_squared_error(y_train_reg, y_pred_rf_train)),
    'RMSE_test': np.sqrt(mean_squared_error(y_test_reg, y_pred_rf_test)),
    'R2_train': r2_score(y_train_reg, y_pred_rf_train),
    'R2_test': r2_score(y_test_reg, y_pred_rf_test)
}

print("\nRandom Forest Ergebnisse:")
print(rf_metrics)

# 3. Vergleich der Modelle
comparison_df = pd.DataFrame({
    'Linear Regression': linear_metrics,
    'Random Forest': rf_metrics
}).round(4)

print("\nModellvergleich:")
print(comparison_df)

Linear Regression Ergebnisse:
{'MAE_train': 2788.069646606067, 'MAE_test': 2575.0124958563665, 'RMSE_train': np.float64(3751.438570907443), 'RMSE_test': np.float64(3405.4235643488764), 'R2_train': 0.8383709172902298, 'R2_test': 0.8518102520046786}


/Users/alexanderdall/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/alexanderdall/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/alexanderdall/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Users/alexanderdall/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/alexanderdall/PyCharmMiscProject/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/alexanderda


Random Forest Ergebnisse:
{'MAE_train': 1326.243838134515, 'MAE_test': 2382.7992723066272, 'RMSE_train': np.float64(1785.4863598182335), 'RMSE_test': np.float64(3279.770356631265), 'R2_train': 0.9633868661751097, 'R2_test': 0.8625442985765073}

Modellvergleich:
            Linear Regression  Random Forest
MAE_train           2788.0696      1326.2438
MAE_test            2575.0125      2382.7993
RMSE_train          3751.4386      1785.4864
RMSE_test           3405.4236      3279.7704
R2_train               0.8384         0.9634
R2_test                0.8518         0.8625
